**Pipeline Logging Utility**

Called via `%run` by all pipeline notebooks. Writes job status to `airline.gold.pipeline_logs`.

In [0]:
from pyspark.sql import Row
from datetime import datetime

LOG_TABLE = "airline.gold.pipeline_logs"

def init_log_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            job_name      STRING,
            layer         STRING,
            run_id        STRING,
            start_time    TIMESTAMP,
            end_time      TIMESTAMP,
            rows_read     LONG,
            rows_written  LONG,
            status        STRING,
            error_message STRING
        )
        USING DELTA
        TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    print(f"✅ Log table ready → {LOG_TABLE}")


def write_log(job_name, layer, run_id, start_time,
              rows_read=0, rows_written=0,
              status="success", error_message=""):
    log_row = spark.createDataFrame([Row(
        job_name      = job_name,
        layer         = layer,
        run_id        = run_id,
        start_time    = start_time,
        end_time      = datetime.utcnow(),
        rows_read     = int(rows_read),
        rows_written  = int(rows_written),
        status        = status,
        error_message = error_message
    )])
    log_row.write.format("delta").mode("append").saveAsTable(LOG_TABLE)
    print(f"✅ Log written → {job_name} | {status} | rows_written={rows_written}")


print("✅ Logging functions loaded")